In [1]:
import os

import importlib
import json
import pandas as pd
import random
import time

from datetime import date, datetime
from dotenv import load_dotenv
from openai import OpenAI

import parsing
import evaluation
import exporting

import pipeline
from pipeline import process_run, should_reuse_api_call, load_run_config
from evaluation import evaluation_v1, evaluation_v2, evaluation_v3, evaluation_v4

In [2]:
RUN_TYPE = "train"

In [3]:
random.seed(42)

In [4]:
importlib.reload(parsing)
from parsing import parse_output

importlib.reload(evaluation)
from evaluation import evaluation_v1, evaluation_v2, evaluation_v3, evaluation_v4

importlib.reload(exporting)
from exporting import export_run

# BioRED Train & GT Loading & Parsing

### Load Key

In [5]:
load_dotenv()
print(os.getenv("OPEN_AI_TEST_KEY")[:15])

sk-proj-4WDSBIA


In [6]:
client = OpenAI(
    api_key=os.getenv("OPEN_AI_TEST_KEY")
)

In [7]:
import modules.biored_loading as biored

In [8]:
importlib.reload(biored)

<module 'modules.biored_loading' from '/Users/wes/Desktop/MSc-Data-Science/MSc-Data-Science/MAST7865 - Data Science Project/biomedical_kg_thesis/exploration/modules/biored_loading.py'>

### Load BioRED

In [9]:
biored_train_sample = biored.load_biored("../data/processed/biored/br_train.csv")

### Get Ground Truths
* Optional sample functionality

In [10]:
biored_train_gts_filtered = biored.get_biored_gts(biored_train_sample, "../data/processed/biored/br_train_entity_relations.csv")

### Export Filtered GTS

In [11]:
biored.export_biored(biored_train_gts_filtered, "biored_train_gts.csv")

Exported dataset ground truths to '../data/filtered/biored_train_gts.csv'.


In [12]:
ground_truths = biored.get_gt_entities_relationships(biored_train_gts_filtered)

### Few-Shot Construction

In [13]:
import modules.biored_loading as biored
importlib.reload(biored)

few_shot_path = "../data/few_shot/few_shot_block.txt"

if not os.path.exists(few_shot_path):

    few_shot_block = biored.get_few_shot_examples(
        path_to_train_set="../data/processed/biored/br_train.csv",
        path_to_train_gts="../data/processed/biored/br_train_entity_relations.csv",
        biored_train_samples=biored_train_sample,
        few_shot_export_path=few_shot_path,
        run_type=RUN_TYPE
    )
    print(f"Generated and exported new few-shot block to '{few_shot_path}'")
else:
    with open(few_shot_path) as f:
        few_shot_block = f.read()
    print(f"Imported existing few-shot block from '{few_shot_path}'")

print(few_shot_block)

Imported existing few-shot block from '../data/few_shot/few_shot_block.txt'
## EXAMPLE 1:

### Abstract:

Massive urinary protein excretion has been observed after conversion from calcineurin inhibitors to mammalian target of rapamycin (mToR) inhibitors, especially sirolimus, in renal transplant recipients with chronic allograft nephropathy. Because proteinuria is a major predictive factor of poor transplantation outcome, many studies focused on this adverse event during the past years. Whether proteinuria was due to sirolimus or only a consequence of calcineurin inhibitors withdrawal remained unsolved until high range proteinuria has been observed during sirolimus therapy in islet transplantation and in patients who received sirolimus de novo. Podocyte injury and focal segmental glomerulosclerosis have been related to mToR inhibition in some patients, but the pathways underlying these lesions remain hypothetic. We discuss herein the possible mechanisms and the significance of mToR blo

### Import BioRED Extraction Guidelines from `guidelines.txt`

In [14]:
with open("../data/processed/biored/guidelines.txt", "r", encoding="utf-8") as f:
    biored_ext_guidelines = f.read()

biored_ext_guidelines[:100]

'## Guideline of the entities\n\n### General rules\n- Annotate all the spans of all the six concept type'

# LLM API Call

In [15]:
with open("../data/prompt_refinement/prompt_versions.json", "r") as f:
    PROMPTS = json.load(f)

print("Successfully loaded prompts JSON as a dict.")

Successfully loaded prompts JSON as a dict.


In [16]:
RUN = "001"

RUN_NAME = f"run_{RUN}"

cfg = load_run_config(f"run_{RUN}")
PROMPT_VERSION = cfg["prompt_version"]
EVAL_VERSION = cfg["eval_version"]
RUN_NOTES = cfg["notes"]
REUSE_API_CALL = should_reuse_api_call(RUN_NAME)

print(f"Cell ran at {datetime.now().strftime('%Y-%m-%d %H:%M')} for run_{RUN}")
print(f" - Prompt version: {PROMPT_VERSION}")
print(f" - Evalaution version: {EVAL_VERSION}")
print(f" - Notes: '{RUN_NOTES}'")
print(f" - Reuse API Call: '{REUSE_API_CALL}'")
print(f' - Prompt template :\n"""\n{PROMPTS[PROMPT_VERSION]["template"]}\n"""')

Cell ran at 2026-07-31 10:09 for run_001
 - Prompt version: v1
 - Evalaution version: v1
 - Notes: 'Baseline extraction benchmark - does not follow BioRED annotation schema.'
 - Reuse API Call: 'False'
 - Prompt template :
"""
You are a biomedical information extraction system.

Extract all biomedical entities and relationships from the following abstract.

Return ONLY valid JSON.

Schema:

{
	"entities": [
		{
			"text": "entity text",
			"type": "entity type"
		}
	],
	"relationships": [
		{
			"source": "entity 1",
			"relation": "relation type",
			"target": "entity 2"
		}
	]
}

Abstract:

{abstract}
"""


In [ ]:
if not REUSE_API_CALL:

    raw_prompt = PROMPTS[PROMPT_VERSION]["template"]

    start_time = time.perf_counter()

    outputs = []

    for index, row in biored_train_sample.iterrows():
        abstract = row["abstract"]

        prompt = (
            PROMPTS[PROMPT_VERSION]["template"]
            .replace("{abstract}", abstract)
            .replace("{few_shot_block}", few_shot_block)
            .replace("{biored_ext_guidelines}", biored_ext_guidelines)
        )

        response = client.responses.create(
            model="gpt-5.6-luna",
            input=prompt
        )

        outputs.append({
            "pmid": row["pmid"],
            "output": response.output_text
        })

    elapsed_seconds = time.perf_counter() - start_time
    print(f"API calls took {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f} min) for {len(outputs)} abstracts")

    print(f"Successfully ran API call at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f" - Prompt verion: {PROMPT_VERSION}")
    print(f" - Evalaution verion: {EVAL_VERSION}")
    print(f" - Run notes: {RUN_NOTES}")
    print(f" - Elapsed time: {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f}")
else:
    prev_run_id = f"run_{int(RUN) - 1:03d}"

    with open(f"../prompt_runs/{prev_run_id}.json") as f:
        prev_run_log = json.load(f)

    outputs = prev_run_log["outputs"]
    raw_prompt = prev_run_log["prompt"]
    elapsed_seconds = prev_run_log["time_taken"]

    print(f"Reused API call from {prev_run_id}.")

output_dict = {
    "outputs": outputs,
    "time_taken": elapsed_seconds,
    "raw_prompt": raw_prompt,
    "run_notes": RUN_NOTES,
    "prompt_version": PROMPT_VERSION,
    "eval_version": EVAL_VERSION
}

In [ ]:
run_log = process_run(output_dict, ground_truths)

Parsed 35 extractions, 0 failed to parse as JSON


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved run_007 to ../prompt_runs


# Error Sampling

In [ ]:
def sample_errors_strict(predictions, ground_truth, n=20, label="items"):
    false_positives = list(predictions - ground_truth)
    false_negatives = list(ground_truth - predictions)

    fp_sample = random.sample(false_positives, min(n, len(false_positives)))
    fn_sample = random.sample(false_negatives, min(n, len(false_negatives)))

    print(f"--- {label}: False Positives (predicted, not in ground truth) ---")
    print(f"Sampled {len(fp_sample)} of {len(false_positives)} total FPs\n")
    for item in fp_sample:
        print(" ", item)

    print(f"\n--- {label}: False Negatives (in ground truth, not predicted) ---")
    print(f"Sampled {len(fn_sample)} of {len(false_negatives)} total FNs\n")
    for item in fn_sample:
        print(" ", item)

    return fp_sample, fn_sample

In [ ]:
from evaluation import COSINE_THRESHOLD

def sample_errors_cosine(predictions, ground_truth, match_fn, embeddings, threshold=COSINE_THRESHOLD, n=20, label="items"):
    matched_predictions = set()
    matched_gt = set()

    for p in predictions:
        best_score = -1
        best_g = None

        for g in ground_truth:
            if g in matched_gt:
                continue
            is_match, score = match_fn(p, g, embeddings, threshold)
            if is_match and score > best_score:
                best_score = score
                best_g = g

        if best_g is not None:
            matched_gt.add(best_g)
            matched_predictions.add(p)

    false_positives = list(predictions - matched_predictions)
    false_negatives = list(ground_truth - matched_gt)

    fp_sample = random.sample(false_positives, min(n, len(false_positives)))
    fn_sample = random.sample(false_negatives, min(n, len(false_negatives)))

    print(f"--- {label}: False Positives (predicted, no cosine match in ground truth) ---")
    print(f"Sampled {len(fp_sample)} of {len(false_positives)} total FPs\n")
    for item in fp_sample:
        print(" ", item)

    print(f"\n--- {label}: False Negatives (in ground truth, no cosine match in predictions) ---")
    print(f"Sampled {len(fn_sample)} of {len(false_negatives)} total FNs\n")
    for item in fn_sample:
        print(" ", item)

    return fp_sample, fn_sample

In [ ]:
from evaluation import relationship_match_cosine, entity_match_cosine, build_embedding_lookup

relationship_embeddings = build_embedding_lookup(
    run_info["predictions_relationships"], ground_truth_relationships, text_indices=[1, 3]
)

NameError: name 'run_info' is not defined

In [ ]:
relation_fp_sample, relation_fn_sample = sample_errors_cosine(
    run_info["predictions_relationships"],
    ground_truth_relationships,
    relationship_match_cosine,
    relationship_embeddings,
    n=100,
    label="Relationships"
)

--- Relationships: False Positives (predicted, no cosine match in ground truth) ---
Sampled 100 of 907 total FPs

  (16120104, 'missense mutation', 'associated_with', 'advanced sleep phase syndrome (asps)')
  (19521089, "parkinson's disease (pd)", 'causes', 'bradykinesia')
  (20431083, 'antiplatelet users', 'have increased frequency of', 'cerebral microbleeds')
  (21163864, 't704c polymorphism', 'has allele', 't allele')
  (16200390, 'two-stage dna pooling design', 'screened', 'genetic loci')
  (18768591, 'plasma aldosterone', 'increases', 'sgk1 protein expression')
  (15099351, 'one patient', 'double_heterozygous_for', 'n157k')
  (20510337, 'cisplatin', 'induces_overexpression_of', 'inducible nitric oxide synthase')
  (28512644, 'il-1beta', 'variation_associated_with', 'sod2 t2734c')
  (28512644, 'il-9', 'serum_level_associated_with', 'predisposition to erysipelas')
  (16200390, 'two-stage dna pooling design', 'screened', 'aadc')
  (19918264, 'arggly genotype', 'higher_incidence_in', 

In [ ]:
entity_embeddings = build_embedding_lookup(
    run_info["predictions_entities"], ground_truth_entities, text_indices=[1]
)

entity_fp_sample, entity_fn_sample = sample_errors_cosine(
    run_info["predictions_entities"],
    ground_truth_entities,
    entity_match_cosine,
    entity_embeddings,
    n=100,
    label="Entities"
)

--- Entities: False Positives (predicted, no cosine match in ground truth) ---
Sampled 100 of 853 total FPs

  (10491763, 'pro75', 'amino acid residue')
  (24309294, 'population spikes', 'electrophysiological measurement')
  (18827003, 'obesity', 'Disease')
  (15686794, 'intravenous amiodarone loading', 'drug administration')
  (28512644, 'higher body part localizations', 'anatomical localization')
  (24477591, 'polish population', 'population')
  (24309294, 'central nervous system', 'organ system')
  (19108278, 'chronotropic responses', 'physiological response')
  (19108278, 'cats', 'animal model')
  (16288197, 'optn', 'gene')
  (18808529, 'isoproterenol-induced myocardial damage', 'disease or injury')
  (16200390, 'allele frequencies', 'genetic measurement')
  (17975693, 'fluoroquinolones', 'drug class')
  (21163864, 'missense mutation', 'genetic variant')
  (21163864, 'methionine to threonine substitution', 'amino acid substitution')
  (17975693, 'ciprofloxacin', 'drug')
  (24309294